# SYMNet (Fast Symmetric Diffeomorphic CNN) — OCT→MRI alignment on the I58 dataset

Companion to the **CPD** notebook, for the model
[*Fast Symmetric Diffeomorphic Image Registration with Convolutional Neural Networks*](https://github.com/cwmok/Fast-Symmetric-Diffeomorphic-Image-Registration-with-Convolutional-Neural-Networks)
(Mok & Chung, CVPR 2020).

**Same as the CPD notebook:** data loading (Drive mount + identical paths), the low-memory block-averaging NIfTI loader, Otsu specimen masks, the CPD-style **affine** reference (octreg if supplied, else a SimpleITK Mattes-MI affine) used here to *pre-align* OCT→MRI, the **Dice** and **surface-residual (mm)** metrics, and the red-contour multi-planar overlays.

**Different from the CPD notebook (because SYMNet is deformable, not affine):**
- SYMNet predicts a dense *diffeomorphic deformation field*, so the affine-only columns (pose-distance, rotation, scale, Det A) are replaced by **deformation-field metrics**: % folding (negative Jacobian), mean/max deformation (mm), and field smoothness.
- We run **test-time optimization**: SYMNet is trained *from scratch on this single OCT–MRI pair* using the repo's own symmetric loss (NCC + magnitude + Jacobian + smoothness). The pretrained weights are brain-MRI→MRI and do not transfer to cross-modality brainstem data.
- Pipeline per *variant*: pre-align OCT→MRI with an affine **start**, resample both onto a shared ×16 grid, normalize, then let SYMNet refine deformably. Variants sweep the affine **start** and the **smoothness** weight — mirroring the CPD `STARTS` × `W_LIST` grid.

> GPU strongly recommended (Runtime → Change runtime type → GPU).


In [ ]:
!pip -q install SimpleITK nibabel scikit-learn pandas scipy matplotlib
import torch, os
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('ok')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/DATA_DIR'     # shortcut to the shared folder
!ls -la "$DATA_DIR/Datasets" 

In [ ]:
# ---------------- CONFIG ----------------
# Data paths — identical to the CPD notebook
MRI_PATH      = f'{DATA_DIR}/Datasets/I58_brainstem_MRI_cropped_to_OCT.nii.gz'
OCT_PATH      = f'{DATA_DIR}/Datasets/I58_Brainstem_mus_Slice_full_20um_corr.nii.gz'
MRI_MASK_PATH = None            # specimen mask in MRI space (None -> Otsu + largest component)
OCT_MASK_PATH = None            # specimen mask in OCT space
OCTREG_AFFINE_PATH = None       # octreg affine: ITK transform (.tfm/.mat/.txt) or 4x4 text matrix (physical mm, LPS)
OCTREG_DIRECTION   = 'fixed_to_moving'   # 'fixed_to_moving' (MRI->OCT, ITK resampling convention) or 'moving_to_fixed'
PRELOAD_MM     = 0.25           # volumes are block-averaged to this while loading (20 um OCT cannot fit in RAM)
TARGET_SPACING = 0.5            # mm, isotropic working grid for SYMNet, Dice, surface residual and overlays

# SYMNet / test-time-training params
START_CHANNEL = 7               # network width (repo: 7 or 8)
RANGE_FLOW    = 100             # flow scaling used throughout the repo
TIME_STEP     = 7               # scaling-and-squaring steps for the diffeomorphic integration
N_ITERS       = 800             # test-time optimization iterations per variant
LR            = 1e-3            # Adam LR (single-pair instance optimization; repo uses 1e-4 for population training)
NCC_WIN       = 7               # local normalized cross-correlation window
W_MAGNITUDE   = 0.1             # loss3 weight  (repo --magnitude)
W_JDET        = 100.0           # loss4 weight  (repo --local_ori, negative-Jacobian penalty)
PAD_MULTIPLE  = 16              # each grid dim must be divisible by this (network downsamples x16)
SEED          = 0

# Variant grid (mirrors CPD's STARTS x W_LIST)
STARTS        = ['affine', 'com', 'header']   # affine pre-alignment of OCT->MRI before SYMNet
SMOOTH_W_LIST = [3.0]                          # loss5 smoothness weight(s) (repo --smooth)

# status thresholds
DICE_GAIN_OK  = 0.02            # 'succeeded' if final Dice exceeds affine-init Dice by at least this
FOLD_WARN_PCT = 0.5            # % voxels with negative Jacobian above this -> 'high folding' flag
DICE_LOW      = 0.30            # final Dice below this -> 'low overlap' flag

OUT = '/content/results_symnet'; os.makedirs(OUT, exist_ok=True)
import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
import os, time, numpy as np, SimpleITK as sitk, pandas as pd
from scipy import ndimage as ndi
import gzip, nibabel as nib, gc
def load_lowmem(path, preload_mm, slab_mem_mb=300):
    # Single sequential pass through the (gzipped) file: block-average slab by slab, RAM stays ~ slab size.
    r = sitk.ImageFileReader(); r.SetFileName(path); r.ReadImageInformation()
    size, spacing, origin, direction = r.GetSize(), np.array(r.GetSpacing()), np.array(r.GetOrigin()), r.GetDirection()
    if len(size) < 3: raise ValueError(f'{path}: expected a 3D volume, got {len(size)}D {size}')
    size = np.array(size[:3])
    f = np.maximum(1, np.floor(preload_mm / spacing + 1e-6)).astype(int)
    print(f'{os.path.basename(path)}: size {tuple(size)}, spacing {np.round(spacing,4)} mm, {np.prod(size)/1e6:.0f}M voxels -> block factor {f}')
    px = nib.load(path).dataobj                      # header only; no voxel data is read here
    dtype = px.dtype; off = int(px.offset); slope, inter = px.slope, px.inter
    slope = 1.0 if slope is None or not np.isfinite(slope) or slope == 0 else float(slope); inter = 0.0 if inter is None or not np.isfinite(inter) else float(inter)
    nx_, ny_, nz_ = [int(v) for v in size]; ox, oy, oz = nx_ // f[0], ny_ // f[1], nz_ // f[2]
    nx, ny = ox * f[0], oy * f[1]
    out = np.zeros((ox, oy, oz), np.float32)
    per_z = nx_ * ny_ * dtype.itemsize
    kb = max(1, int(slab_mem_mb * 1e6 // (per_z * 2)) // int(f[2]))        # output slices per slab
    opener = gzip.open if str(path).endswith('.gz') else open
    t0 = time.time(); k0 = 0
    with opener(path, 'rb') as fh:
        fh.seek(off)
        while k0 < oz:
            k = min(kb, oz - k0); zs = k * int(f[2])
            buf = fh.read(per_z * zs)
            slab = np.frombuffer(buf, dtype=dtype, count=nx_ * ny_ * zs).reshape((nx_, ny_, zs), order='F')[:nx, :ny]
            slab = slab.astype(np.float32) if (slope == 1.0 and inter == 0.0) else slab.astype(np.float32) * slope + inter
            np.nan_to_num(slab, copy=False)
            out[:, :, k0:k0 + k] = slab.reshape(ox, f[0], oy, f[1], k, f[2]).mean(axis=(1, 3, 5))
            del slab, buf; k0 += k
            el = time.time() - t0; print(f'  {100*k0/oz:5.1f}%  elapsed {el/60:5.1f} min  ETA {el*(oz-k0)/max(k0,1)/60:5.1f} min', end='\r')
    print()
    img = sitk.GetImageFromArray(np.transpose(out, (2, 1, 0)))
    D = np.array(direction).reshape(3, 3)
    img.SetSpacing(tuple(spacing * f)); img.SetDirection(direction)
    img.SetOrigin(tuple(origin + D @ ((f - 1) / 2.0 * spacing)))
    return img


def otsu_mask(img):
    m = sitk.OtsuThreshold(sitk.Cast(img, sitk.sitkFloat32), 0, 1)
    m = sitk.BinaryMorphologicalClosing(m, [3]*3)
    cc = sitk.RelabelComponent(sitk.ConnectedComponent(m)); m = cc == 1
    return sitk.BinaryFillhole(m)
rdm = lambda p: sitk.Cast(sitk.ReadImage(p, sitk.sitkFloat32) > 0, sitk.sitkUInt8)

t0 = time.time()
mri, oct_ = load_lowmem(MRI_PATH, PRELOAD_MM), load_lowmem(OCT_PATH, PRELOAD_MM)
print('load time %.1f min' % ((time.time() - t0) / 60))
mri_m = rdm(MRI_MASK_PATH) if MRI_MASK_PATH else otsu_mask(mri)
oct_m = rdm(OCT_MASK_PATH) if OCT_MASK_PATH else otsu_mask(oct_)
print('MRI', mri.GetSize(), mri.GetSpacing(), '| OCT', oct_.GetSize(), oct_.GetSpacing())
print('mask voxels  MRI', int(sitk.GetArrayFromImage(mri_m).sum()), ' OCT', int(sitk.GetArrayFromImage(oct_m).sum()))

In [ ]:
# ======================================================================
# SYMNet model + transforms + losses, embedded verbatim from the repo's
# Code/Models.py (cwmok, CVPR 2020).  .cuda() calls are made device-aware.
# ======================================================================
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np

class SYMNet(nn.Module):
    def __init__(self, in_channel, n_classes, start_channel):
        self.in_channel = in_channel; self.n_classes = n_classes; self.start_channel = start_channel
        bias_opt = True
        super(SYMNet, self).__init__()
        self.eninput = self.encoder(self.in_channel, self.start_channel, bias=bias_opt)
        self.ec1 = self.encoder(self.start_channel, self.start_channel, bias=bias_opt)
        self.ec2 = self.encoder(self.start_channel, self.start_channel * 2, stride=2, bias=bias_opt)
        self.ec3 = self.encoder(self.start_channel * 2, self.start_channel * 2, bias=bias_opt)
        self.ec4 = self.encoder(self.start_channel * 2, self.start_channel * 4, stride=2, bias=bias_opt)
        self.ec5 = self.encoder(self.start_channel * 4, self.start_channel * 4, bias=bias_opt)
        self.ec6 = self.encoder(self.start_channel * 4, self.start_channel * 8, stride=2, bias=bias_opt)
        self.ec7 = self.encoder(self.start_channel * 8, self.start_channel * 8, bias=bias_opt)
        self.ec8 = self.encoder(self.start_channel * 8, self.start_channel * 16, stride=2, bias=bias_opt)
        self.ec9 = self.encoder(self.start_channel * 16, self.start_channel * 8, bias=bias_opt)
        self.dc1 = self.encoder(self.start_channel * 8 + self.start_channel * 8, self.start_channel * 8, kernel_size=3, stride=1, bias=bias_opt)
        self.dc2 = self.encoder(self.start_channel * 8, self.start_channel * 4, kernel_size=3, stride=1, bias=bias_opt)
        self.dc3 = self.encoder(self.start_channel * 4 + self.start_channel * 4, self.start_channel * 4, kernel_size=3, stride=1, bias=bias_opt)
        self.dc4 = self.encoder(self.start_channel * 4, self.start_channel * 2, kernel_size=3, stride=1, bias=bias_opt)
        self.dc5 = self.encoder(self.start_channel * 2 + self.start_channel * 2, self.start_channel * 4, kernel_size=3, stride=1, bias=bias_opt)
        self.dc6 = self.encoder(self.start_channel * 4, self.start_channel * 2, kernel_size=3, stride=1, bias=bias_opt)
        self.dc7 = self.encoder(self.start_channel * 2 + self.start_channel * 1, self.start_channel * 2, kernel_size=3, stride=1, bias=bias_opt)
        self.dc8 = self.encoder(self.start_channel * 2, self.start_channel * 2, kernel_size=3, stride=1, bias=bias_opt)
        self.dc9 = self.outputs(self.start_channel * 2, self.n_classes, kernel_size=5, stride=1, padding=2, bias=False)
        self.dc10 = self.outputs(self.start_channel * 2, self.n_classes, kernel_size=5, stride=1, padding=2, bias=False)
        self.up1 = self.decoder(self.start_channel * 8, self.start_channel * 8)
        self.up2 = self.decoder(self.start_channel * 4, self.start_channel * 4)
        self.up3 = self.decoder(self.start_channel * 2, self.start_channel * 2)
        self.up4 = self.decoder(self.start_channel * 2, self.start_channel * 2)
    def encoder(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, batchnorm=False):
        if batchnorm:
            return nn.Sequential(nn.Conv3d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias), nn.BatchNorm3d(out_channels), nn.ReLU())
        return nn.Sequential(nn.Conv3d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias), nn.ReLU())
    def decoder(self, in_channels, out_channels, kernel_size=2, stride=2, padding=0, output_padding=0, bias=True):
        return nn.Sequential(nn.ConvTranspose3d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, output_padding=output_padding, bias=bias), nn.ReLU())
    def outputs(self, in_channels, out_channels, kernel_size=3, stride=1, padding=0, bias=False, batchnorm=False):
        if batchnorm:
            return nn.Sequential(nn.Conv3d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias), nn.BatchNorm3d(out_channels), nn.Tanh())
        return nn.Sequential(nn.Conv3d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias), nn.Softsign())
    def forward(self, x, y):
        x_in = torch.cat((x, y), 1)
        e0 = self.eninput(x_in); e0 = self.ec1(e0)
        e1 = self.ec2(e0); e1 = self.ec3(e1)
        e2 = self.ec4(e1); e2 = self.ec5(e2)
        e3 = self.ec6(e2); e3 = self.ec7(e3)
        e4 = self.ec8(e3); e4 = self.ec9(e4)
        d0 = torch.cat((self.up1(e4), e3), 1); d0 = self.dc1(d0); d0 = self.dc2(d0)
        d1 = torch.cat((self.up2(d0), e2), 1); d1 = self.dc3(d1); d1 = self.dc4(d1)
        d2 = torch.cat((self.up3(d1), e1), 1); d2 = self.dc5(d2); d2 = self.dc6(d2)
        d3 = torch.cat((self.up4(d2), e0), 1); d3 = self.dc7(d3); d3 = self.dc8(d3)
        f_xy = self.dc9(d3); f_yx = self.dc10(d3)
        return f_xy, f_yx

class SpatialTransform(nn.Module):
    def __init__(self): super(SpatialTransform, self).__init__()
    def forward(self, x, flow, sample_grid):
        sample_grid = sample_grid + flow
        st = sample_grid.size()
        sample_grid = torch.stack([
            (sample_grid[0, :, :, :, 0] - ((st[3]-1)/2)) / (st[3]-1) * 2,
            (sample_grid[0, :, :, :, 1] - ((st[2]-1)/2)) / (st[2]-1) * 2,
            (sample_grid[0, :, :, :, 2] - ((st[1]-1)/2)) / (st[1]-1) * 2], dim=3).unsqueeze(0)
        return torch.nn.functional.grid_sample(x, sample_grid, mode='bilinear', align_corners=True)

class SpatialTransformNearest(nn.Module):
    def __init__(self): super(SpatialTransformNearest, self).__init__()
    def forward(self, x, flow, sample_grid):
        sample_grid = sample_grid + flow
        st = sample_grid.size()
        sample_grid = torch.stack([
            (sample_grid[0, :, :, :, 0] - ((st[3]-1)/2)) / (st[3]-1) * 2,
            (sample_grid[0, :, :, :, 1] - ((st[2]-1)/2)) / (st[2]-1) * 2,
            (sample_grid[0, :, :, :, 2] - ((st[1]-1)/2)) / (st[1]-1) * 2], dim=3).unsqueeze(0)
        return torch.nn.functional.grid_sample(x, sample_grid, mode='nearest', align_corners=True)

class DiffeomorphicTransform(nn.Module):
    def __init__(self, time_step=7): super(DiffeomorphicTransform, self).__init__(); self.time_step = time_step
    def forward(self, velocity, sample_grid, range_flow):
        flow = velocity / (2.0 ** self.time_step)
        st = sample_grid.size()
        for _ in range(self.time_step):
            g = sample_grid + (flow.permute(0, 2, 3, 4, 1) * range_flow)
            g = torch.stack([
                (g[0, :, :, :, 0] - ((st[3]-1)/2)) / (st[3]-1) * 2,
                (g[0, :, :, :, 1] - ((st[2]-1)/2)) / (st[2]-1) * 2,
                (g[0, :, :, :, 2] - ((st[1]-1)/2)) / (st[1]-1) * 2], dim=3).unsqueeze(0)
            flow = flow + F.grid_sample(flow, g, mode='bilinear', align_corners=True)
        return flow

class CompositionTransform(nn.Module):
    def __init__(self): super(CompositionTransform, self).__init__()
    def forward(self, flow_1, flow_2, sample_grid, range_flow):
        st = sample_grid.size()
        g = sample_grid + (flow_2.permute(0, 2, 3, 4, 1) * range_flow)
        g = torch.stack([
            (g[0, :, :, :, 0] - ((st[3]-1)/2)) / (st[3]-1) * 2,
            (g[0, :, :, :, 1] - ((st[2]-1)/2)) / (st[2]-1) * 2,
            (g[0, :, :, :, 2] - ((st[1]-1)/2)) / (st[1]-1) * 2], dim=3).unsqueeze(0)
        return F.grid_sample(flow_1, g, mode='bilinear', align_corners=True) + flow_2

def smoothloss(y_pred):
    dy = torch.abs(y_pred[:, :, 1:, :, :] - y_pred[:, :, :-1, :, :])
    dx = torch.abs(y_pred[:, :, :, 1:, :] - y_pred[:, :, :, :-1, :])
    dz = torch.abs(y_pred[:, :, :, :, 1:] - y_pred[:, :, :, :, :-1])
    return (torch.mean(dx * dx) + torch.mean(dy * dy) + torch.mean(dz * dz)) / 3.0

def JacboianDet(y_pred, sample_grid):
    J = y_pred + sample_grid
    dy = J[:, 1:, :-1, :-1, :] - J[:, :-1, :-1, :-1, :]
    dx = J[:, :-1, 1:, :-1, :] - J[:, :-1, :-1, :-1, :]
    dz = J[:, :-1, :-1, 1:, :] - J[:, :-1, :-1, :-1, :]
    Jdet0 = dx[:, :, :, :, 0] * (dy[:, :, :, :, 1] * dz[:, :, :, :, 2] - dy[:, :, :, :, 2] * dz[:, :, :, :, 1])
    Jdet1 = dx[:, :, :, :, 1] * (dy[:, :, :, :, 0] * dz[:, :, :, :, 2] - dy[:, :, :, :, 2] * dz[:, :, :, :, 0])
    Jdet2 = dx[:, :, :, :, 2] * (dy[:, :, :, :, 0] * dz[:, :, :, :, 1] - dy[:, :, :, :, 1] * dz[:, :, :, :, 0])
    return Jdet0 - Jdet1 + Jdet2

def neg_Jdet_loss(y_pred, sample_grid):
    return torch.mean(F.relu(-1.0 * JacboianDet(y_pred, sample_grid)))

def magnitude_loss(flow_1, flow_2):
    num_ele = torch.numel(flow_1)
    return (torch.abs(torch.sum(torch.abs(flow_1)) - torch.sum(torch.abs(flow_2)))) / num_ele

class NCC(torch.nn.Module):
    """local (windowed) normalized cross correlation; from VoxelMorph."""
    def __init__(self, win=7, eps=1e-5):
        super(NCC, self).__init__(); self.win_raw = win; self.eps = eps; self.win = win
    def forward(self, I, J):
        ndims = 3; win_size = self.win_raw; self.win = [self.win_raw] * ndims
        weight = torch.ones((1, 1, win_size, win_size, win_size), device=I.device, requires_grad=False)
        conv_fn = F.conv3d
        I2, J2, IJ = I * I, J * J, I * J
        I_sum = conv_fn(I, weight, padding=int(win_size / 2)); J_sum = conv_fn(J, weight, padding=int(win_size / 2))
        I2_sum = conv_fn(I2, weight, padding=int(win_size / 2)); J2_sum = conv_fn(J2, weight, padding=int(win_size / 2))
        IJ_sum = conv_fn(IJ, weight, padding=int(win_size / 2))
        win_size = np.prod(self.win); u_I = I_sum / win_size; u_J = J_sum / win_size
        cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
        I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
        J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size
        cc = cross * cross / (I_var * J_var + self.eps)
        return -1.0 * torch.mean(cc)

def generate_grid(imgshape):
    x = np.arange(imgshape[0]); y = np.arange(imgshape[1]); z = np.arange(imgshape[2])
    grid = np.rollaxis(np.array(np.meshgrid(z, y, x)), 0, 4)
    grid = np.swapaxes(grid, 0, 2); grid = np.swapaxes(grid, 1, 2)
    return grid

def imgnorm(img):
    i_max = np.max(img); i_min = np.min(img)
    return (img - i_min) / (i_max - i_min + 1e-8)

print('SYMNet model defined.')

In [ ]:
from itertools import product
# ---------- geometry helpers (all transforms are (A, t): p' = A p + t) ----------
def geom(img): return np.array(img.GetSpacing()), np.array(img.GetDirection()).reshape(3, 3), np.array(img.GetOrigin())
def idx_to_phys(img, idx): sp, D, o = geom(img); return o + (D @ (np.asarray(idx, float) * sp).T).T
def phys_to_idx(img, P):   sp, D, o = geom(img); return (np.linalg.inv(D) @ (np.asarray(P) - o).T).T / sp
def centre(img): return idx_to_phys(img, ((np.array(img.GetSize()) - 1) / 2.0)[None])[0]
def vox_idx(mask_img, surface):
    m = sitk.GetArrayFromImage(mask_img) > 0
    if surface: m = m & ~ndi.binary_erosion(m)
    return np.argwhere(m)[:, ::-1].astype(float)                 # (z,y,x) -> (x,y,z)
def com(mask_img): return idx_to_phys(mask_img, vox_idx(mask_img, False).mean(0)[None])[0]
inv_aff  = lambda A, t: (np.linalg.inv(A), -np.linalg.inv(A) @ t)
apply_aff = lambda A, t, P: P @ A.T + t
def polar_R(A): U, _, Vt = np.linalg.svd(A); return U @ Vt
rot_angle = lambda Ra, Rb: float(np.degrees(np.arccos(np.clip((np.trace(Ra.T @ Rb) - 1) / 2, -1, 1))))
def to_sitk(A, t):
    tx = sitk.AffineTransform(3); tx.SetMatrix(A.ravel().tolist()); tx.SetTranslation(t.tolist()); return tx
def tx_to_affine(tx):
    t0 = np.array(tx.TransformPoint((0., 0., 0.)))
    return np.stack([np.array(tx.TransformPoint(tuple(e))) - t0 for e in np.eye(3)], 1), t0

mri_com, oct_com = com(mri_m), com(oct_)

# ---------- affine reference (octreg if supplied, else SimpleITK Mattes-MI) : MRI(fixed) -> OCT(moving) ----------
def read_ref(path, direction):
    try: A, t = tx_to_affine(sitk.ReadTransform(path))
    except Exception: M = np.loadtxt(path); A, t = M[:3, :3], M[:3, 3]
    return inv_aff(A, t) if direction == 'moving_to_fixed' else (A, t)
if OCTREG_AFFINE_PATH:
    REF_NAME = 'octreg'; A_ref, t_ref = read_ref(OCTREG_AFFINE_PATH, OCTREG_DIRECTION)
else:
    REF_NAME = 'MI-affine (SimpleITK, not octreg)'
    init = sitk.CenteredTransformInitializer(mri, oct_, sitk.AffineTransform(3), sitk.CenteredTransformInitializerFilter.GEOMETRY)
    R_ = sitk.ImageRegistrationMethod(); R_.SetMetricAsMattesMutualInformation(32); R_.SetMetricSamplingStrategy(R_.RANDOM); R_.SetMetricSamplingPercentage(0.2)
    R_.SetInterpolator(sitk.sitkLinear); R_.SetOptimizerAsGradientDescent(1.0, 300, estimateLearningRate=R_.EachIteration)
    R_.SetOptimizerScalesFromPhysicalShift(); R_.SetShrinkFactorsPerLevel([4, 2, 1]); R_.SetSmoothingSigmasPerLevel([2, 1, 0])
    R_.SetInitialTransform(init, inPlace=False); A_ref, t_ref = tx_to_affine(R_.Execute(mri, oct_))
print('affine reference:', REF_NAME); print(np.round(A_ref, 4), np.round(t_ref, 3))

# ---------- shared working grid: MRI geometry at TARGET_SPACING, padded so each dim % PAD_MULTIPLE == 0 ----------
base = np.array([int(round(s * o / TARGET_SPACING)) for s, o in zip(mri.GetSize(), mri.GetSpacing())])
padded = np.array([int(np.ceil(b / PAD_MULTIPLE)) * PAD_MULTIPLE for b in base])
pad = padded - base
sp_ = [TARGET_SPACING] * 3
_D = np.array(mri.GetDirection()).reshape(3, 3)
grid = sitk.Image([int(v) for v in padded], sitk.sitkFloat32)
grid.SetSpacing(sp_); grid.SetDirection(mri.GetDirection())
grid.SetOrigin(tuple(np.array(mri.GetOrigin()) - _D @ (pad / 2.0 * TARGET_SPACING)))   # centre the padding
IMG_SHAPE = tuple(int(v) for v in sitk.GetArrayFromImage(grid).shape)                   # (z, y, x) for the network
print('working grid size', grid.GetSize(), '-> network shape (z,y,x)', IMG_SHAPE, '| spacing', grid.GetSpacing())

# MRI (fixed) resampled onto the grid, plus masks
mri_g  = sitk.Resample(mri,   grid, sitk.Transform(), sitk.sitkLinear, 0.0)
mri_mg = sitk.Resample(mri_m, grid, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
PLANES_D = np.array(grid.GetDirection()).reshape(3, 3)

In [ ]:
# ---------- affine 'start': fixed(MRI)->moving(OCT) transform used to pre-align OCT onto the grid ----------
def start_affine(name):
    if name == 'header':  return np.eye(3), np.zeros(3)                       # identity (physical coords)
    if name == 'com':     return np.eye(3), (oct_com - mri_com)               # centre-of-mass translation
    if name == 'affine':  return A_ref, t_ref                                 # CPD-style MI/octreg affine
    raise ValueError(name)

def prepare_moving(name):
    A_fm, t_fm = start_affine(name)                                           # MRI->OCT (resampling transform)
    oct_on_grid  = sitk.Resample(oct_,  grid, to_sitk(A_fm, t_fm), sitk.sitkLinear, 0.0)
    octm_on_grid = sitk.Resample(oct_m, grid, to_sitk(A_fm, t_fm), sitk.sitkNearestNeighbor, 0)
    return oct_on_grid, octm_on_grid

def to_tensor(sitk_img, norm=True):
    a = sitk.GetArrayFromImage(sitk_img).astype(np.float32)                   # (z,y,x)
    if norm: a = imgnorm(a)
    return torch.from_numpy(a[None, None]).float().to(DEVICE)                 # (1,1,z,y,x)

mri_t   = to_tensor(mri_g, norm=True)
mri_mt  = torch.from_numpy((sitk.GetArrayFromImage(mri_mg) > 0).astype(np.float32)[None, None]).to(DEVICE)
print('fixed (MRI) tensor', tuple(mri_t.shape))

In [ ]:
from scipy.spatial import cKDTree
# ---------- metrics shared with the CPD notebook (computed on the working grid, in mm) ----------
def dice_np(a, b): a = a > 0.5; b = b > 0.5; return float(2 * (a & b).sum() / max(a.sum() + b.sum(), 1))
def surface_residual_mm(a_mask, b_mask, spacing=TARGET_SPACING):
    # symmetric mean surface-to-surface distance between two binary masks (array order z,y,x)
    def surf_pts(m):
        m = m > 0.5; s = m & ~ndi.binary_erosion(m); return np.argwhere(s).astype(float) * spacing
    pa, pb = surf_pts(a_mask), surf_pts(b_mask)
    if len(pa) == 0 or len(pb) == 0: return float('nan')
    d1 = cKDTree(pb).query(pa)[0].mean(); d2 = cKDTree(pa).query(pb)[0].mean()
    return float((d1 + d2) / 2)

# ---------- deformation-field metrics (replace CPD's affine-only columns) ----------
def field_metrics(F_mov2fix, grid_t):
    # F_mov2fix: (1,3,z,y,x) displacement in grid-voxel units (already scaled by range_flow)
    disp = F_mov2fix.permute(0, 2, 3, 4, 1)                                   # (1,z,y,x,3)
    Jdet = JacboianDet(disp, grid_t).detach().cpu().numpy()                   # (z-1,y-1,x-1)
    fold_pct = 100.0 * float((Jdet <= 0).mean())
    mag_vox = torch.sqrt((disp ** 2).sum(-1)).detach().cpu().numpy()[0]       # per-voxel magnitude (voxels)
    mag_mm = mag_vox * TARGET_SPACING
    smooth = float(smoothloss(F_mov2fix).detach().cpu().numpy())
    return dict(fold_pct=fold_pct, mean_def_mm=float(mag_mm.mean()), max_def_mm=float(mag_mm.max()), smoothness=smooth)

print('metric helpers ready.')

In [ ]:
import time
# ---------- one SYMNet test-time optimization on the single (MRI, OCT) pair ----------
grid_np = generate_grid(IMG_SHAPE)
grid_t  = torch.from_numpy(grid_np[None]).float().to(DEVICE)                  # (1,z,y,x,3)

def run_variant(start, smooth_w, n_iters=N_ITERS, lr=LR, verbose=True):
    t_start = time.time()
    torch.manual_seed(SEED); np.random.seed(SEED)
    oct_on_grid, octm_on_grid = prepare_moving(start)
    Y = to_tensor(oct_on_grid, norm=True)                                     # moving = affine-aligned OCT
    Y_mask = torch.from_numpy((sitk.GetArrayFromImage(octm_on_grid) > 0).astype(np.float32)[None, None]).to(DEVICE)
    X = mri_t                                                                 # fixed = MRI

    model = SYMNet(2, 3, START_CHANNEL).to(DEVICE)
    transform   = SpatialTransform().to(DEVICE)
    transform_n = SpatialTransformNearest().to(DEVICE)
    diff_tr = DiffeomorphicTransform(time_step=TIME_STEP).to(DEVICE)
    com_tr  = CompositionTransform().to(DEVICE)
    for p in transform.parameters(): p.requires_grad = False
    ncc = NCC(win=NCC_WIN)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # affine-init (pre-SYMNet) baseline
    dice_init = dice_np(sitk.GetArrayFromImage(octm_on_grid), sitk.GetArrayFromImage(mri_mg))
    resid_init = surface_residual_mm(sitk.GetArrayFromImage(octm_on_grid), sitk.GetArrayFromImage(mri_mg))

    loss_hist = []
    model.train()
    for it in range(n_iters):
        F_xy, F_yx = model(X, Y)
        F_X_Y_half = diff_tr(F_xy, grid_t, RANGE_FLOW); F_Y_X_half = diff_tr(F_yx, grid_t, RANGE_FLOW)
        F_X_Y_half_inv = diff_tr(-F_xy, grid_t, RANGE_FLOW); F_Y_X_half_inv = diff_tr(-F_yx, grid_t, RANGE_FLOW)
        X_Y_half = transform(X, F_X_Y_half.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t)
        Y_X_half = transform(Y, F_Y_X_half.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t)
        F_X_Y = com_tr(F_X_Y_half, F_Y_X_half_inv, grid_t, RANGE_FLOW)
        F_Y_X = com_tr(F_Y_X_half, F_X_Y_half_inv, grid_t, RANGE_FLOW)
        X_Y = transform(X, F_X_Y.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t)
        Y_X = transform(Y, F_Y_X.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t)
        loss1 = ncc(X_Y_half, Y_X_half)
        loss2 = ncc(Y, X_Y) + ncc(X, Y_X)
        loss3 = magnitude_loss(F_X_Y_half * RANGE_FLOW, F_Y_X_half * RANGE_FLOW)
        loss4 = neg_Jdet_loss(F_X_Y.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t) + neg_Jdet_loss(F_Y_X.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t)
        loss5 = smoothloss(F_xy * RANGE_FLOW) + smoothloss(F_yx * RANGE_FLOW)
        loss = loss1 + loss2 + W_MAGNITUDE * loss3 + W_JDET * loss4 + smooth_w * loss5
        opt.zero_grad(); loss.backward(); opt.step()
        loss_hist.append(loss.item())
        if verbose and (it % max(1, n_iters // 10) == 0 or it == n_iters - 1):
            print(f'  [{start}|smo{smooth_w}] iter {it:4d}/{n_iters}  loss {loss.item():.4f}  (sim {loss2.item():.4f})', end='\r')
    if verbose: print()

    # final fields / warps (OCT -> MRI space uses F_Y_X, exactly as repo Test_SYMNet.py 'warped_B')
    model.eval()
    with torch.no_grad():
        F_xy, F_yx = model(X, Y)
        F_X_Y_half = diff_tr(F_xy, grid_t, RANGE_FLOW); F_Y_X_half = diff_tr(F_yx, grid_t, RANGE_FLOW)
        F_X_Y_half_inv = diff_tr(-F_xy, grid_t, RANGE_FLOW); F_Y_X_half_inv = diff_tr(-F_yx, grid_t, RANGE_FLOW)
        F_Y_X = com_tr(F_Y_X_half, F_X_Y_half_inv, grid_t, RANGE_FLOW)
        F_Y_X_scaled = F_Y_X * RANGE_FLOW                                     # displacement in grid-voxel units
        warped_oct      = transform(Y, F_Y_X.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t)
        warped_oct_mask = transform_n(Y_mask, F_Y_X.permute(0, 2, 3, 4, 1) * RANGE_FLOW, grid_t)
    warped_oct_np  = warped_oct.cpu().numpy()[0, 0]
    warped_mask_np = warped_oct_mask.cpu().numpy()[0, 0]
    mri_mask_np    = sitk.GetArrayFromImage(mri_mg)

    dice_final  = dice_np(warped_mask_np, mri_mask_np)
    resid_final = surface_residual_mm(warped_mask_np, mri_mask_np)
    fm = field_metrics(F_Y_X_scaled, grid_t)
    secs = time.time() - t_start

    # status (analogous to CPD's classify)
    if not np.isfinite(dice_final) or dice_final < 1e-6:
        status = 'failed (degenerate)'
    else:
        flags = []
        if fm['fold_pct'] > FOLD_WARN_PCT: flags.append('high folding')
        if dice_final < DICE_LOW: flags.append('low overlap')
        if dice_final >= dice_init + DICE_GAIN_OK and not flags: status = 'succeeded (improved over affine)'
        elif dice_final < dice_init - DICE_GAIN_OK: status = 'degraded vs affine; ' + '; '.join(flags) if flags else 'degraded vs affine'
        else: status = '; '.join(flags) if flags else 'near affine (little deformable gain)'

    row = {'Tool': 'SYMNet (test-time trained)', 'Start / affine init':
           {'affine': f'CPD affine ({REF_NAME})', 'com': 'centre of mass', 'header': 'header (identity)'}[start],
           'Smooth w': smooth_w, 'Status': status,
           'Dice init (affine)': dice_init, 'Dice (warped OCT vs MRI)': dice_final,
           'Surf resid init (mm)': resid_init, 'Surface residual (mm)': resid_final,
           'Folding % (neg Jac)': fm['fold_pct'], 'Mean deform (mm)': fm['mean_def_mm'],
           'Max deform (mm)': fm['max_def_mm'], 'Field smoothness': fm['smoothness'], 'Time (s)': secs}
    extra = dict(warped_oct=warped_oct_np, warped_mask=warped_mask_np, loss_hist=loss_hist,
                 oct_affine=sitk.GetArrayFromImage(oct_on_grid), octm_affine=sitk.GetArrayFromImage(octm_on_grid),
                 F_Y_X=F_Y_X_scaled.detach().cpu().numpy()[0],
                 dice_final=dice_final, start=start, smooth_w=smooth_w)
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return row, extra
print('run_variant() ready.')

In [ ]:
rows, store = [], {}
for smooth_w in SMOOTH_W_LIST:
    for start in STARTS:
        key = f'{start}|smo{smooth_w}'
        try:
            r_, ex_ = run_variant(start, smooth_w); rows.append(r_); store[key] = ex_
        except Exception as e:
            rows.append({'Tool': 'SYMNet', 'Start / affine init': start, 'Smooth w': smooth_w, 'Status': f'failed: {type(e).__name__}: {e}'})
            print('FAILED', key, e)
        last = rows[-1]
        print(f"{key:18s} {last.get('Status')}  | Dice {last.get('Dice init (affine)', float('nan')):.3f}->{last.get('Dice (warped OCT vs MRI)', float('nan')):.3f}"
              f"  resid {last.get('Surface residual (mm)', float('nan')):.2f}mm  fold {last.get('Folding % (neg Jac)', float('nan')):.2f}%  {last.get('Time (s)', 0):.1f}s")

res = pd.DataFrame(rows); res.insert(0, 'Reference affine', REF_NAME)
res.to_csv(f'{OUT}/symnet_results.csv', index=False)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 60)
res.round(3)

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
# Overlay style identical to CPD: warped OCT in grayscale, MRI specimen boundary in red, on the working grid.
mb = sitk.GetArrayFromImage(mri_mg) > 0                                       # array order (z,y,x)
D_ = np.array(grid.GetDirection()).reshape(3, 3)
ax_of = lambda phys_axis: 2 - int(np.argmax(np.abs(D_[phys_axis])))           # array axis for physical axis 0=L-R,1=A-P,2=S-I
PLANES = {'axial': ax_of(2), 'coronal': ax_of(1), 'sagittal': ax_of(0)}
sup_sgn = int(np.sign(D_[2, int(np.argmax(np.abs(D_[2])))]))
def slice_idx(plane, where='mid'):
    ax = PLANES[plane]; c = np.argwhere(mb)[:, ax]
    if where == 'mid': return int(np.median(c))
    top = int(c.max() if (plane == 'axial' and sup_sgn > 0) or plane != 'axial' else c.min())
    return int(np.clip(top - sup_sgn * round(2 / grid.GetSpacing()[0]), 0, mb.shape[ax] - 1))
def panel(a, w, plane, sl, title):
    take = lambda v: np.take(v, sl, axis=PLANES[plane])
    img = take(w); pos = w[w > 0]; lo_, hi_ = (np.percentile(pos, 1), np.percentile(pos, 99.5)) if pos.size else (0, 1)
    a.imshow(img, cmap='gray', vmin=lo_, vmax=hi_, origin='lower', interpolation='nearest')
    a.contour(take(mb.astype(float)), [0.5], colors='red', linewidths=1.0)
    a.set_title(title, fontsize=7); a.axis('off')

# items: affine-init baseline (per start) + SYMNet-warped per variant
items = []
for start in STARTS:
    k0 = f'{start}|smo{SMOOTH_W_LIST[0]}'
    if k0 in store: items.append((f'AFFINE INIT: {start}', store[k0]['oct_affine']))
for key in store: items.append((f'SYMNet: {key}', store[key]['warped_oct']))

ncol = 4; nrow = int(np.ceil(len(items) / ncol))
for tag, plane, where in [('axial_mid', 'axial', 'mid'), ('axial_superior', 'axial', 'sup'), ('sagittal_mid', 'sagittal', 'mid'), ('coronal_mid', 'coronal', 'mid')]:
    sl = slice_idx(plane, where); fig, axs = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 4 * nrow)); axs = np.atleast_2d(axs)
    for a in axs.ravel(): a.axis('off')
    for a, (name, vol) in zip(axs.ravel(), items): panel(a, vol, plane, sl, name)
    plt.suptitle(f'{tag}: warped OCT (gray), MRI boundary (red)'); plt.tight_layout(); plt.savefig(f'{OUT}/overlay_contact_{tag}.png', dpi=140); plt.show()

# one image per variant: axial mid | axial superior | sagittal | coronal
for name, vol in items:
    fig, axs = plt.subplots(1, 4, figsize=(14, 5))
    for a, (plane, where) in zip(axs, [('axial', 'mid'), ('axial', 'sup'), ('sagittal', 'mid'), ('coronal', 'mid')]):
        panel(a, vol, plane, slice_idx(plane, where), f'{name}\n{plane} {where}')
    plt.tight_layout(); plt.savefig(f"{OUT}/overlay_{name.replace(':', '').replace('|', '_').replace(' ', '_')}.png", dpi=140); plt.close(fig)
print('overlays saved to', OUT)

In [ ]:
# Deformation-specific diagnostics: training-loss curves, deformation-magnitude map, Jacobian-determinant map.
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for key, ex in store.items(): ax.plot(ex['loss_hist'], label=key, lw=1)
ax.set_xlabel('iteration'); ax.set_ylabel('total loss'); ax.set_title('SYMNet test-time training loss'); ax.legend(fontsize=7); plt.tight_layout()
plt.savefig(f'{OUT}/loss_curves.png', dpi=140); plt.show()

# choose the variant with the highest final Dice
best_key = max(store, key=lambda k: store[k]['dice_final'])
ex = store[best_key]
Fz = ex['F_Y_X']                                   # (3, z, y, x) grid-voxel displacement
mag_mm = np.sqrt((Fz ** 2).sum(0)) * TARGET_SPACING
disp = np.moveaxis(Fz, 0, -1)[None]                # (1,z,y,x,3)
Jdet = JacboianDet(torch.from_numpy(disp).float(), torch.from_numpy(grid_np[None]).float()).numpy()[0]

sl = slice_idx('axial', 'mid')
fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
axs[0].imshow(np.take(ex['warped_oct'], sl, axis=PLANES['axial']), cmap='gray', origin='lower'); axs[0].set_title(f'warped OCT — {best_key}'); axs[0].axis('off')
im1 = axs[1].imshow(np.take(mag_mm, sl, axis=PLANES['axial']), cmap='magma', origin='lower'); axs[1].set_title('deformation magnitude (mm)'); axs[1].axis('off'); fig.colorbar(im1, ax=axs[1], fraction=0.046)
jsl = min(sl, Jdet.shape[PLANES['axial']] - 1)
im2 = axs[2].imshow(np.take(Jdet, jsl, axis=PLANES['axial']), cmap='RdBu', vmin=-1, vmax=3, origin='lower'); axs[2].set_title('Jacobian det (blue<0 = folding)'); axs[2].axis('off'); fig.colorbar(im2, ax=axs[2], fraction=0.046)
plt.tight_layout(); plt.savefig(f'{OUT}/deformation_maps_{best_key.replace("|","_")}.png', dpi=140); plt.show()

In [ ]:
from IPython.display import Image, display
# Best SYMNet variant by final Dice (CPD picked best by pose distance; here Dice is the overlap analogue).
best_key = max(store, key=lambda k: store[k]['dice_final'])
best_idx = res['Dice (warped OCT vs MRI)'].idxmax()
best_row = res.loc[best_idx]
print('Best SYMNet configuration:')
print(f"- Start / affine init : {best_row['Start / affine init']}")
print(f"- Smooth weight       : {best_row['Smooth w']}")
print(f"- Dice (affine -> SYMNet): {best_row['Dice init (affine)']:.3f} -> {best_row['Dice (warped OCT vs MRI)']:.3f}")
print(f"- Surface residual    : {best_row['Surface residual (mm)']:.3f} mm")
print(f"- Folding (neg Jac)   : {best_row['Folding % (neg Jac)']:.3f} %")
print(f"- Mean / max deform   : {best_row['Mean deform (mm)']:.3f} / {best_row['Max deform (mm)']:.3f} mm\n")

name = f"SYMNet: {best_key}"
img_path = f"{OUT}/overlay_{name.replace(':', '').replace('|', '_').replace(' ', '_')}.png"
print('Multi-planar overlay for the best variant:')
display(Image(filename=img_path))